In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import torch

In [ ]:
philox_m = [0xD2511F53, 0xCD9E8D57]
philox_w = [0x9E3779B9, 0xBB67AE85]

two_pow32_inv = np.array([2.3283064e-10], dtype=np.float32)
two_pow32_inv_2pi = np.array([2.3283064e-10 * 6.2831855], dtype=np.float32)


def uint32(x):
    """Converts (N,) np.uint64 array into (2, N) np.unit32 array."""
    return x.view(np.uint32).reshape(-1, 2).transpose(1, 0)


def philox4_round(counter, key):
    """A single round of the Philox 4x32 random number generator."""

    v1 = uint32(counter[0].astype(np.uint64) * philox_m[0])
    v2 = uint32(counter[2].astype(np.uint64) * philox_m[1])

    counter[0] = v2[1] ^ counter[1] ^ key[0]
    counter[1] = v2[0]
    counter[2] = v1[1] ^ counter[3] ^ key[1]
    counter[3] = v1[0]


def philox4_32(counter, key, rounds=10):
    """Generates 32-bit random numbers using the Philox 4x32 random number generator.

    Parameters:
        counter (numpy.ndarray): A 4xN array of 32-bit integers representing the counter values (offset into generation).
        key (numpy.ndarray): A 2xN array of 32-bit integers representing the key values (seed).
        rounds (int): The number of rounds to perform.

    Returns:
        numpy.ndarray: A 4xN array of 32-bit integers containing the generated random numbers.
    """

    for _ in range(rounds - 1):
        philox4_round(counter, key)

        key[0] = key[0] + philox_w[0]
        key[1] = key[1] + philox_w[1]

    philox4_round(counter, key)
    return counter


def box_muller(x, y):
    """Returns just the first out of two numbers generated by Box–Muller transform algorithm."""
    u = x * two_pow32_inv + two_pow32_inv / 2
    v = y * two_pow32_inv_2pi + two_pow32_inv_2pi / 2

    s = np.sqrt(-2.0 * np.log(u))

    r1 = s * np.sin(v)
    return r1.astype(np.float32)


class Generator:
    """RNG that produces same outputs as torch.randn(..., device='cuda') on CPU"""

    def __init__(self, seed):
        self.seed = seed
        self.offset = 0

    def randn(self, shape):
        """Generate a sequence of n standard normal random variables using the Philox 4x32 random number generator and the Box-Muller transform."""

        n = 1
        for x in shape:
            n *= x

        counter = np.zeros((4, n), dtype=np.uint32)
        counter[0] = self.offset
        counter[2] = np.arange(n, dtype=np.uint32)  # up to 2^32 numbers can be generated - if you want more you'd need to spill into counter[3]
        self.offset += 1

        key = np.empty(n, dtype=np.uint64)
        key.fill(self.seed)
        key = uint32(key)

        g = philox4_32(counter, key)

        return box_muller(g[0], g[1]).reshape(shape)  # discard g[2] and g[3]

In [9]:
jax.config.update("jax_enable_x64", True)

In [15]:
torch.manual_seed(0)
print(torch.randn((4,4), device='cuda'))

tensor([[-0.9247, -0.4253, -2.6438,  0.1452],
        [-0.1209, -0.5797, -0.6229, -0.3284],
        [-1.0745, -0.3631, -1.6711,  2.2655],
        [ 0.3117, -0.1842,  1.2866,  1.1820]], device='cuda:0')


In [13]:
philox_randn(shape=(3,4), seed=0)

Array([[ 0.83222705,  0.5104095 , -2.5331168 ,  0.2281982 ],
       [ 0.83222705,  0.5104095 , -2.5331168 ,  0.2281982 ],
       [ 0.83222705,  0.5104095 , -2.5331168 ,  0.2281982 ]],      dtype=float32)

In [ ]:
print(torch.randn((3,4), device='cpu'))

In [16]:
gen = Generator(seed=0)
print(gen.randn((4,4)))

[[-0.9246624  -0.42534396 -2.6438458   0.14518388]
 [-0.12086651 -0.5797257  -0.6228513  -0.32838714]
 [-1.074542   -0.3631441  -1.6710504   2.265505  ]
 [ 0.31168503 -0.18418905  1.2866426   1.1819526 ]]
